## 🎯 Learning Outcomes

### By the end of this module, you will be able to:

- Understand the **JSON ↔ Python** type mapping
- Use **`json.loads`** and **`json.dumps`** for string-based JSON operations
- Use **`json.load`** and **`json.dump`** for file-based JSON operations
- Implement **custom encoders/decoders** for non-native types (e.g., datetime, Decimal, sets)
- Perform **nested JSON traversal** and convert API responses to DataFrames

---

In [1]:
import json
import pandas as pd

## Part 1 — JSON ↔ Python Types

| JSON | Python |
|---|---|
| object | `dict` |
| array | `list` |
| string | `str` |
| number (int / float) | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

JSON has **no** dates, tuples, sets, NaN, or comments. Dates are usually transmitted as **ISO-8601 strings** (e.g. `"2026-05-09T11:30:00"`) and parsed afterwards.

In [2]:
# -- Parse a JSON object -------------------------------------------------------
text = '{"name": "Ada", "skills": ["python", "numpy"], "age": 30, "active": true, "manager": null}'
data = json.loads(text)

print("type:", type(data))
print("data:", data)
print()
print("skills[0] :", data['skills'][0])
print("active    :", data['active'])
print("manager   :", data['manager'])     # None  (was null in JSON)

type: <class 'dict'>
data: {'name': 'Ada', 'skills': ['python', 'numpy'], 'age': 30, 'active': True, 'manager': None}

skills[0] : python
active    : True
manager   : None


## Part 2 — Strings: `loads` and `dumps`

The `s` in **loads** / **dumps** stands for **string**. Versions without it (`json.load` / `json.dump`) work on **file objects** instead.

In [3]:
# -- Python -> JSON string -----------------------------------------------------
record = {
    'user':   'ada',
    'scores': [95, 88, 91],
    'when':   '2026-05-09',
}

# Dump as compact (default)
compact = json.dumps(record)
print("compact:")
print(compact)

# Dump as pretty-printed
pretty = json.dumps(record, indent=2, ensure_ascii=False)
print("\npretty:")
print(pretty)

compact:
{"user": "ada", "scores": [95, 88, 91], "when": "2026-05-09"}

pretty:
{
  "user": "ada",
  "scores": [
    95,
    88,
    91
  ],
  "when": "2026-05-09"
}


In [4]:
# -- JSON string -> Python -----------------------------------------------------
parsed = json.loads(pretty)
print("round-trip equal:", parsed == record)

# Common dumps options:
#   indent=2          -> pretty-print
#   sort_keys=True    -> deterministic output (great for diffs / hashing)
#   ensure_ascii=False-> keep non-ASCII characters as themselves (Arabic, emoji)
print("\nsorted, non-ASCII kept:")
print(json.dumps({'aleph': 'aleph', 'name': 'Lina'},
                 indent=2, sort_keys=True, ensure_ascii=False))

round-trip equal: True

sorted, non-ASCII kept:
{
  "aleph": "aleph",
  "name": "Lina"
}


## Part 3 — Files: `load` and `dump`

Always open with `encoding='utf-8'`. Use `with open(...)` so the file is closed even if an error happens.

In [5]:
record = {'user': 'ada', 'scores': [95, 88, 91]}

# -- Write ---------------------------------------------------------------------
with open('/tmp/record.json', 'w', encoding='utf-8') as f:
    json.dump(record, f, indent=2)

# -- Read back -----------------------------------------------------------------
with open('/tmp/record.json', encoding='utf-8') as f:
    loaded = json.load(f)

print("loaded:", loaded)
print("\nfile content on disk:")
print(open('/tmp/record.json').read())

loaded: {'user': 'ada', 'scores': [95, 88, 91]}

file content on disk:
{
  "user": "ada",
  "scores": [
    95,
    88,
    91
  ]
}


## Part 3b — Loading a JSON File

Parts 1–3 showed the mechanics of `loads`/`dumps`/`load`/`dump`. Now we apply them to a **real file**: `students.json` — a student-records payload with a top-level metadata envelope and a nested `scores` object inside every student record.

Place `students.json` in the same folder as this notebook before running.

| Cell | What it shows |
|---|---|
| **Load & inspect** | `json.load`, navigate the envelope, understand `null` → `None` |
| **Manual flatten** | loop → extract nested fields → `pd.DataFrame` |
| **`pd.json_normalize`** | the one-liner shortcut for the same job |

In [6]:
import json
import pandas as pd

# -- Load students.json from disk -----------------------------------------
# Make sure students.json is in the same folder as this notebook
with open('students.json', encoding='utf-8') as f:
    data = json.load(f)

# -- Inspect the envelope (top-level keys) --------------------------------
print('Top-level keys :', list(data.keys()))
print('Course         :', data['course'])
print('Semester       :', data['semester'])
print('Student count  :', len(data['students']))
print()

# -- Inspect one record to understand the shape ---------------------------
first = data['students'][0]
print('First student  :', first)
print()
print('Nested scores  :', first['scores'])    # dict inside a dict
print('Math score     :', first['scores']['math'])
print('Notes (null→None):', first['notes'])   # JSON null becomes Python None

Top-level keys : ['course', 'semester', 'students']
Course         : Data Science Fundamentals
Semester       : Spring 2026
Student count  : 10

First student  : {'id': 1, 'name': 'Ali Hassan', 'age': 18, 'scores': {'math': 88, 'science': 92, 'english': 75}, 'passed': True, 'notes': None}

Nested scores  : {'math': 88, 'science': 92, 'english': 75}
Math score     : 88
Notes (null→None): None


In [7]:
import pandas as pd

# -- Manual flatten: drill into 'scores' for each student -------------
# Use this pattern when you only need specific nested fields
rows = []
for s in data['students']:
    rows.append({
        'id'     : s['id'],
        'name'   : s['name'],
        'age'    : s['age'],
        'math'   : s['scores']['math'],      # drill one level deeper
        'science': s['scores']['science'],
        'english': s['scores']['english'],
        'passed' : s['passed'],
        'notes'  : s['notes'],               # None stays as NaN in DataFrame
    })

df = pd.DataFrame(rows)
print(f'DataFrame: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Pass rate : {df["passed"].mean():.0%}')
print(f'Top scorer: {df.loc[df["math"].idxmax(), "name"]} '
      f'(math={df["math"].max()})')
df

DataFrame: 10 rows × 8 columns
Pass rate : 70%
Top scorer: Fatima Al-Zahra (math=95)


,id,name,age,math,science,english,passed,notes
0,1,Ali Hassan,18,88,92,75,True,None
1,2,Sara Al-Mutairi,19,72,68,84,True,strong english
2,3,Khalid Nasser,18,55,60,58,False,needs support
3,4,Fatima Al-Zahra,20,95,97,91,True,None
4,5,Omar Bin Saleh,19,43,50,62,False,at risk
5,6,Noura Al-Rashidi,18,78,74,80,True,None
6,7,Youssef Khalil,21,66,70,55,True,None
7,8,Maryam Ibrahim,19,90,88,93,True,top performer
8,9,Tariq Al-Jabri,20,38,45,49,False,at risk
9,10,Hessa Al-Mansoori,18,82,79,88,True,None


In [8]:
# -- pd.json_normalize: the one-liner alternative ---------------------
# Automatically flattens ALL nested dicts; nested keys become col_subkey
df_norm = pd.json_normalize(data['students'], sep='_')

print('Columns after normalize:', df_norm.columns.tolist())
print()

# 'scores_math', 'scores_science', 'scores_english' are created automatically
df_norm[['name', 'scores_math', 'scores_science', 'scores_english',
         'passed', 'notes']]

Columns after normalize: ['id', 'name', 'age', 'passed', 'notes', 'scores_math', 'scores_science', 'scores_english']



,name,scores_math,scores_science,scores_english,passed,notes
0,Ali Hassan,88,92,75,True,None
1,Sara Al-Mutairi,72,68,84,True,strong english
2,Khalid Nasser,55,60,58,False,needs support
3,Fatima Al-Zahra,95,97,91,True,None
4,Omar Bin Saleh,43,50,62,False,at risk
5,Noura Al-Rashidi,78,74,80,True,None
6,Youssef Khalil,66,70,55,True,None
7,Maryam Ibrahim,90,88,93,True,top performer
8,Tariq Al-Jabri,38,45,49,False,at risk
9,Hessa Al-Mansoori,82,79,88,True,None


> **Key takeaways — loading a JSON file**
>
> | Task | Code |
> |---|---|
> | Write a dict to a file | `json.dump(data, f, indent=2, ensure_ascii=False)` |
> | Read a file into a dict | `json.load(f)` (always open with `encoding='utf-8'`) |
> | Navigate an envelope | `data['students']` — drill by key |
> | JSON `null` → Python | `None` (and `NaN` inside a DataFrame) |
> | Manual flatten | loop → `rows.append({...})` → `pd.DataFrame(rows)` |
> | Auto flatten | `pd.json_normalize(records, sep='_')` |
>
> **When to use which:** manual flatten is clearer when you only need a few fields or want to rename them. `json_normalize` is faster when you want every field and don't mind the auto-generated `parent_child` column names.

## Part 4 — Custom Encoders for non-native types

`datetime`, `Decimal`, NumPy scalars, and your own classes are not JSON-native. Subclass `JSONEncoder` and override `default()`.

In [9]:
import json
from datetime import datetime
from decimal import Decimal

# -- Custom encoder ------------------------------------------------------------
class MyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()              # 2026-05-09T11:30:00
        if isinstance(obj, Decimal):
            return float(obj)
        if isinstance(obj, set):
            return sorted(obj)
        return super().default(obj)             # let base class raise TypeError

data = {
    'event':  'login',
    'when':   datetime(2026, 5, 9, 11, 30, 0),
    'amount': Decimal('12.50'),
    'tags':   {'auth', 'web', 'security'},
}

print(json.dumps(data, cls=MyEncoder, indent=2))

{
  "event": "login",
  "when": "2026-05-09T11:30:00",
  "amount": 12.5,
  "tags": [
    "auth",
    "security",
    "web"
  ]
}


In [10]:
# -- Custom decoder via object_hook --------------------------------------------
def parse_dates(d):
    """Look for ISO-format date strings and convert them to datetime."""
    for k, v in d.items():
        if isinstance(v, str) and 'T' in v and len(v) >= 19:
            try:
                d[k] = datetime.fromisoformat(v)
            except ValueError:
                pass
    return d

text = '{"event": "login", "when": "2026-05-09T11:30:00"}'
obj = json.loads(text, object_hook=parse_dates)

print("obj         :", obj)
print("type(when)  :", type(obj['when']).__name__)

obj         : {'event': 'login', 'when': datetime.datetime(2026, 5, 9, 11, 30)}
type(when)  : datetime


## Part 5 — Nested JSON and API Responses

Real APIs return nested JSON. The standard recipe:

1. **Inspect** the structure
2. **Drill in** to the relevant list
3. **Flatten** each record (manually, or with `pd.json_normalize`)
4. Build a **DataFrame**

We'll simulate a typical API payload (no network needed).

In [11]:
# -- A typical API payload -----------------------------------------------------
api_payload = {
    'status': 'ok',
    'meta':   {'page': 1, 'per_page': 3, 'total': 3},
    'data': {
        'users': [
            {'id': 1, 'name': 'Ada',
             'city': {'name': 'Dhahran', 'country': 'SA'},
             'tags': ['admin']},
            {'id': 2, 'name': 'Omar',
             'city': {'name': 'Riyadh', 'country': 'SA'},
             'tags': ['user']},
            {'id': 3, 'name': 'Lina',
             'city': {'name': 'Jeddah', 'country': 'SA'},
             'tags': ['user', 'beta']},
        ]
    }
}

# -- 1. Drill in ---------------------------------------------------------------
users = api_payload['data']['users']
print(f"{len(users)} users found")

# -- 2. Manual flatten ---------------------------------------------------------
rows = [
    {'id': u['id'],
     'name': u['name'],
     'city': u['city']['name'],
     'country': u['city']['country']}
    for u in users
]
pd.DataFrame(rows)

3 users found


,id,name,city,country
0,1,Ada,Dhahran,SA
1,2,Omar,Riyadh,SA
2,3,Lina,Jeddah,SA


In [12]:
# -- 3. pd.json_normalize handles nested fields automatically ------------------
pd.json_normalize(users, sep='_')

,id,name,tags,city_name,city_country
0,1,Ada,[admin],Dhahran,SA
1,2,Omar,[user],Riyadh,SA
2,3,Lina,"[user, beta]",Jeddah,SA


In [13]:
# -- 4. Pull the records list AND attach top-level metadata via meta= ----------
pd.json_normalize(
    api_payload,
    record_path=['data', 'users'],
    meta='status',                   # repeats the top-level 'status' on every row
    sep='_',
)

,id,name,tags,city_name,city_country,status
0,1,Ada,[admin],Dhahran,SA,ok
1,2,Omar,[user],Riyadh,SA,ok
2,3,Lina,"[user, beta]",Jeddah,SA,ok


## Part 6 — Calling a Real REST API

Everything in Part 5 used a hard-coded dict. Now we'll talk to an **actual server** over the internet and process whatever it sends back.

We'll use the **PyPI JSON API** — the same service `pip` queries when you install a package. It requires **no account, no key, and no rate-limit** for light use, making it perfect for learning.

| Concept introduced | What you'll see |
|---|---|
| `urllib.request` | Python's built-in HTTP client (no extra installs) |
| HTTP status codes | How to detect success vs error |
| Response headers | `Content-Type`, rate-limit hints |
| Nested JSON drilling | `data['info']`, `data['urls']` |
| `pd.json_normalize` on live data | Flatten real nested objects |
| Release history | Looping over a dict-of-lists response |

**Endpoint pattern:** `https://pypi.org/pypi/{package}/json`

In [14]:
import urllib.request
import json
import pandas as pd

# -- Make the HTTP GET request ------------------------------------------
URL = 'https://pypi.org/pypi/requests/json'

req = urllib.request.Request(URL, headers={'User-Agent': 'learning-demo/1.0'})

with urllib.request.urlopen(req) as response:
    status  = response.status                    # 200 = OK
    ctype   = response.headers['Content-Type']   # should be application/json
    raw     = response.read()                     # bytes from the wire

print(f'HTTP status  : {status}')
print(f'Content-Type : {ctype}')
print(f'Bytes received: {len(raw):,}')

# -- Decode bytes -> str -> Python dict ---------------------------------
data = json.loads(raw)

print(f'\nTop-level keys : {list(data.keys())}')
print(f'Type of data   : {type(data).__name__}')

HTTP status  : 200
Content-Type : application/json
Bytes received: 181,513

Top-level keys : ['info', 'last_serial', 'ownership', 'releases', 'urls', 'vulnerabilities']
Type of data   : dict


In [15]:
# -- Drill into the 'info' block — package metadata --------------------
info = data['info']

print(f"Package   : {info['name']}")
print(f"Version   : {info['version']}")
print(f"Summary   : {info['summary']}")
print(f"License   : {info['license_expression']}")
print(f"Py req.   : {info['requires_python']}")
print(f"Home page : {info['home_page']}")
print()

# -- 'classifiers' is a list of strings — slice to first 5 -------------
print('Classifiers (first 5):')
for c in info['classifiers'][:5]:
    print(' ', c)

# -- 'requires_dist' lists runtime dependencies -----------------------
print('\nDependencies (first 5):')
for dep in (info['requires_dist'] or [])[:5]:
    print(' ', dep)

Package   : requests
Version   : 2.34.2
Summary   : Python HTTP for Humans.
License   : None
Py req.   : >=3.10
Home page : None

Classifiers (first 5):
  Development Status :: 5 - Production/Stable
  Environment :: Web Environment
  Intended Audience :: Developers
  License :: OSI Approved :: Apache Software License
  Natural Language :: English

Dependencies (first 5):
  charset_normalizer<4,>=2
  idna<4,>=2.5
  urllib3<3,>=1.26
  certifi>=2023.5.7
  PySocks!=1.5.7,>=1.5.6; extra == "socks"


In [16]:
# -- 'urls' is a list of dicts, one per distribution file --------------
print(f"Latest release has {len(data['urls'])} distribution file(s)")
print(f"Keys in each file record: {list(data['urls'][0].keys())}")
print()

# -- Flatten the 'urls' list into a DataFrame with json_normalize ------
files_df = pd.json_normalize(data['urls'])

# Keep only the most useful columns
cols = ['filename', 'packagetype', 'python_version',
        'size', 'upload_time_iso_8601', 'url']
print(files_df[cols].to_string(index=False))
print()
print(f"Largest file: {files_df['size'].max() / 1e6:.2f} MB")

Latest release has 2 distribution file(s)
Keys in each file record: ['comment_text', 'digests', 'downloads', 'filename', 'has_sig', 'md5_digest', 'packagetype', 'python_version', 'requires_python', 'size', 'upload_time', 'upload_time_iso_8601', 'url', 'yanked', 'yanked_reason']

                        filename packagetype python_version   size        upload_time_iso_8601                                                                                                                                         url
requests-2.34.2-py3-none-any.whl bdist_wheel            py3  73075 2026-05-14T19:25:26.443000Z https://files.pythonhosted.org/packages/a0/f4/c67b0b3f1b9245e8d266f0f112c500d50e5b4e83cb6f3b71b6528104182a/requests-2.34.2-py3-none-any.whl
          requests-2.34.2.tar.gz       sdist         source 142856 2026-05-14T19:25:27.735762Z           https://files.pythonhosted.org/packages/ac/c3/e2a2b89f2d3e2179abd6d00ebd70bff6273f37fb3e0cc209f48b39d00cbf/requests-2.34.2.tar.gz

Largest file: 

### Handling errors gracefully

Networks fail. Packages may not exist. Always wrap API calls in error handling — `urllib.error.HTTPError` carries the status code, so you can react differently to a 404 (not found) vs a 500 (server error).

In [17]:
import urllib.error

def fetch_package(package_name: str) -> dict | None:
    """Fetch PyPI metadata for a package. Returns None on any error."""
    url = f'https://pypi.org/pypi/{package_name}/json'
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'learning-demo/1.0'})
        with urllib.request.urlopen(req, timeout=10) as r:
            return json.loads(r.read())
    except urllib.error.HTTPError as e:
        print(f'HTTP {e.code} for "{package_name}" — {e.reason}')
    except urllib.error.URLError as e:
        print(f'Network error: {e.reason}')
    return None

# -- Test with a real package and a made-up one -----------------------
good = fetch_package('flask')
print('flask version:', good['info']['version'] if good else 'failed')

bad  = fetch_package('this-package-does-not-exist-xyz123')
print('bad package   :', bad)

flask version: 3.1.3
HTTP 404 for "this-package-does-not-exist-xyz123" — Not Found
bad package   : None


### Comparing multiple packages — building a DataFrame from API calls

Loop over a list of packages, call the API for each, extract the fields you need, and assemble a DataFrame. This is the standard pattern for any bulk data-collection task.

In [18]:
packages = ['numpy', 'pandas', 'matplotlib', 'scikit-learn',
            'flask', 'fastapi', 'requests', 'django']

rows = []
for pkg in packages:
    d = fetch_package(pkg)
    if d is None:
        continue
    info = d['info']
    # Find the wheel file, fall back to the first file if none exists
    wheel = next((f for f in d['urls'] if f['packagetype'] == 'bdist_wheel'),
                 d['urls'][0] if d['urls'] else {})
    rows.append({
        'package'        : info['name'],
        'version'        : info['version'],
        'requires_python': info['requires_python'],
        'license'        : (info['license_expression'] or info.get('license') or '')[:30],
        'size_mb'        : round(wheel.get('size', 0) / 1e6, 2),
        'released'       : (wheel.get('upload_time_iso_8601') or '')[:10],
    })

df_pkgs = pd.DataFrame(rows)
df_pkgs.sort_values('size_mb', ascending=False)

,package,version,requires_python,license,size_mb,released
0,numpy,2.4.5,>=3.11,BSD-3-Clause AND 0BSD AND MIT,16.97,2026-05-15
1,pandas,3.0.3,>=3.11,BSD 3-Clause License\n,10.34,2026-05-11
3,scikit-learn,1.8.0,>=3.11,BSD-3-Clause,8.60,2025-12-10
7,Django,6.0.5,>=3.12,BSD-3-Clause,8.37,2026-05-05
2,matplotlib,3.10.9,>=3.10,License agreement for matplotl,8.30,2026-04-24
5,fastapi,0.136.1,>=3.10,MIT,0.12,2026-04-23
4,Flask,3.1.3,>=3.9,BSD-3-Clause,0.10,2026-02-19
6,requests,2.34.2,>=3.10,Apache-2.0,0.07,2026-05-14


### Release history — a dict-of-lists response

The `releases` key maps every version string to the list of files published for it. This is a common JSON pattern — a **dict used as a lookup table** — and it needs a loop to flatten into rows.

In [19]:
# -- Fetch the full releases dict for 'requests' ----------------------
data = fetch_package('requests')
releases = data['releases']          # { '2.28.0': [...files...], ... }

print(f'Total versions in PyPI history: {len(releases)}')

# -- Flatten: one row per version that has at least one file ----------
history = []
for version, files in releases.items():
    if not files:                    # skip yanked/empty versions
        continue
    history.append({
        'version'    : version,
        'upload_date': files[0]['upload_time_iso_8601'][:10],
        'n_files'    : len(files),
        'size_kb'    : round(files[0]['size'] / 1024, 1),
    })

hist_df = (
    pd.DataFrame(history)
      .sort_values('upload_date', ascending=False)
      .reset_index(drop=True)
)

print('Most recent 10 releases:')
print(hist_df.head(10).to_string(index=False))
print()
print(f'First release : {hist_df["upload_date"].min()}')
print(f'Latest release: {hist_df["upload_date"].max()}')
print(f'Years on PyPI : {(pd.to_datetime(hist_df["upload_date"].max()) - pd.to_datetime(hist_df["upload_date"].min())).days // 365}')

Total versions in PyPI history: 163
Most recent 10 releases:
    version upload_date  n_files  size_kb
     2.34.2  2026-05-14        2     71.4
     2.34.1  2026-05-13        2     71.4
     2.34.0  2026-05-11        2     71.3
2.34.0.dev1  2026-05-03        2     71.4
     2.33.1  2026-03-30        2     63.4
     2.33.0  2026-03-25        2     63.5
     2.32.5  2025-08-18        2     63.2
     2.32.4  2025-06-09        2     63.3
     2.32.3  2024-05-29        2     63.4
     2.32.2  2024-05-21        2     62.4

First release : 2011-02-14
Latest release: 2026-05-14
Years on PyPI : 15


> **Key takeaways — real API calls**
>
> | Concept | Code |
> |---|---|
> | Make a GET request | `urllib.request.urlopen(Request(url, headers={...}))` |
> | Check status | `response.status` (200 = OK) |
> | Parse the body | `json.loads(response.read())` |
> | Handle errors | `except urllib.error.HTTPError as e: e.code` |
> | Drill into nested JSON | `data['info']['version']` |
> | Flatten a list of dicts | `pd.json_normalize(data['urls'])` |
> | Flatten a dict-of-lists | `for ver, files in data['releases'].items()` |
> | Build a comparison table | Loop → `rows.append({...})` → `pd.DataFrame(rows)` |
>
> The PyPI API used here is completely free and requires no authentication. The same pattern — `urllib.request` → `json.loads` → drill / normalize → DataFrame — works for any REST API that returns JSON.

---

## Worked Examples

### Example 1 — Round-trip a config file

Read a config file, modify it programmatically, and write it back. A pattern used by every CLI tool that has a `~/.toolrc` JSON.

In [20]:
# -- Initial config ------------------------------------------------------------
config = {
    'theme':   'dark',
    'limits':  {'rate': 100, 'connections': 10},
    'plugins': ['markdown', 'images'],
}

with open('/tmp/config.json', 'w') as f:
    json.dump(config, f, indent=2)

# -- Read, modify, write back --------------------------------------------------
with open('/tmp/config.json') as f:
    cfg = json.load(f)

cfg['theme'] = 'light'                       # change a value
cfg['limits']['rate'] = 200                  # change a nested value
cfg['plugins'].append('latex')               # extend a list

with open('/tmp/config.json', 'w') as f:
    json.dump(cfg, f, indent=2, sort_keys=True)

print(open('/tmp/config.json').read())

{
  "limits": {
    "connections": 10,
    "rate": 200
  },
  "plugins": [
    "markdown",
    "images",
    "latex"
  ],
  "theme": "light"
}


### Example 2 — Compute order totals from a JSON list

You receive a JSON string of orders. Compute total amount, average ticket, and per-currency breakdown.

In [21]:
orders_text = '''[
    {"id": 1, "amount": 25.0,  "currency": "USD"},
    {"id": 2, "amount": 47.5,  "currency": "USD"},
    {"id": 3, "amount": 12.25, "currency": "EUR"},
    {"id": 4, "amount": 87.0,  "currency": "USD"},
    {"id": 5, "amount": 33.0,  "currency": "EUR"}
]'''

orders = json.loads(orders_text)

# -- Aggregate via DataFrame ---------------------------------------------------
df = pd.DataFrame(orders)
print("Per-currency totals:")
print(df.groupby('currency')['amount'].agg(['sum', 'mean', 'count']).round(2))
print(f"\nGrand total : {df['amount'].sum():.2f}")
print(f"Avg ticket  : {df['amount'].mean():.2f}")

Per-currency totals:
             sum   mean  count
currency                      
EUR        45.25  22.62      2
USD       159.50  53.17      3

Grand total : 204.75
Avg ticket  : 40.95


### Example 3 — Flatten a deeply nested catalog into invoice rows

A nested catalog with shop -> items -> price/qty becomes a flat invoice table.

In [22]:
catalog = {
    'shop': 'KFUPM Bookstore',
    'items': [
        {'product': 'Notebook',   'price': 15, 'qty': 4},
        {'product': 'Pen',        'price':  5, 'qty': 12},
        {'product': 'Calculator', 'price': 90, 'qty': 1},
    ],
}

# -- Build invoice -------------------------------------------------------------
invoice = pd.DataFrame(catalog['items'])
invoice['shop']  = catalog['shop']
invoice['total'] = invoice['price'] * invoice['qty']

# -- Reorder columns -----------------------------------------------------------
invoice = invoice[['shop', 'product', 'price', 'qty', 'total']]
invoice.loc['GRAND TOTAL'] = ['', '', '', '', invoice['total'].sum()]
invoice

,shop,product,price,qty,total
0,KFUPM Bookstore,Notebook,15,4,60
1,KFUPM Bookstore,Pen,5,12,60
2,KFUPM Bookstore,Calculator,90,1,90
GRAND TOTAL,,,,,210


### Example 4 — Round-trip a custom class

Save a Python class instance to JSON and load it back. The encoder turns a class into a `dict`; the decoder rebuilds the class.

In [23]:
class User:
    def __init__(self, name: str, email: str, signup_date: datetime):
        self.name = name
        self.email = email
        self.signup_date = signup_date

    def __repr__(self):
        return f"User({self.name!r}, {self.email!r}, signup={self.signup_date.date()})"

# -- Encoder: tag the dict so the decoder knows what to rebuild ----------------
class UserEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, User):
            return {'__type__':    'User',
                    'name':        obj.name,
                    'email':       obj.email,
                    'signup_date': obj.signup_date.isoformat()}
        return super().default(obj)

def user_decoder(d):
    if d.get('__type__') == 'User':
        return User(d['name'], d['email'], datetime.fromisoformat(d['signup_date']))
    return d

# -- Round trip ----------------------------------------------------------------
u_in  = User('Ada', 'ada@example.com', datetime(2026, 1, 15, 9, 0))
text  = json.dumps(u_in, cls=UserEncoder, indent=2)
u_out = json.loads(text, object_hook=user_decoder)

print("JSON form:")
print(text)
print("\nRebuilt:", u_out)
print("type   :", type(u_out).__name__)

JSON form:
{
  "__type__": "User",
  "name": "Ada",
  "email": "ada@example.com",
  "signup_date": "2026-01-15T09:00:00"
}

Rebuilt: User('Ada', 'ada@example.com', signup=2026-01-15)
type   : User


---

## Summary

| Concept | Key Point |
|---------|-----------|
| **JSON ↔ Python Types** | Mapping for objects, arrays, strings, numbers, booleans, and null |
| **`json.loads()`** | Parses a JSON string into a Python object |
| **`json.dumps()`** | Dumps a Python object to a JSON formatted string (with `indent`, `sort_keys` options) |
| **`json.load()`** | Reads JSON data from a file object |
| **`json.dump()`** | Writes a Python object as JSON to a file object |
| **Custom `JSONEncoder`** | Handles non-native Python types (e.g., `datetime`, `Decimal`, `set`) during serialization |
| **`object_hook` in `json.loads()`** | Custom function to process parsed JSON objects during deserialization (e.g., to convert date strings to `datetime` objects) |
| **`pd.json_normalize()`** | Flattens nested JSON structures into a Pandas DataFrame, handling `record_path` and `meta` data |

### Contributed by: Abdulhadi Zubailah